In [ ]:
import numpy as np
from scipy.optimize import root

import plotly.graph_objects as go
from scipy.integrate import solve_ivp

In [42]:
# Equations to find zeroes of
# All 2 trusses
def system_equations(X, epsilons, kappa, kappa_wall):
    X1, X2, X3 = X
    eps1, eps2, eps3 = epsilons

    eq1 = (
        -2 * (1 - (1 + eps1) / np.sqrt(1 + X1**2)) * X1
        - kappa_wall * X1
        - kappa * (X1 - X2)
    )

    eq2 = (
        -2 * (1 - (1 + eps2) / np.sqrt(1 + X2**2)) * X2
        - kappa * (X2 - X1)
        - kappa * (X2 - X3)
    )

    eq3 = (
        -2 * (1 - (1 + eps3) / np.sqrt(1 + X3**2)) * X3
        - kappa_wall * X3
        - kappa * (X3 - X2)
    )

    return np.array([eq1, eq2, eq3])


epsilons = [0.1, 0.1, 0.1]  # Bistable
kappa = 1.0
kappa_wall = 0.0  # Currently just remove wall for now


initial_guesses = []
for x in [-1, 0, 1]:
    for y in [-1, 0, 1]:
        for z in [-1, 0, 1]:
            initial_guesses.append([x, y, z])

unique_solutions = []


# Steady State
print(f"{'Initial Guess':<20} | {'Solver Status':<15} | {'Steady State [X1, X2, X3]'}")
print("-" * 65)

for i, guess in enumerate(initial_guesses):
    sol = root(system_equations, guess, args=(epsilons, kappa, kappa_wall))

    rounded_sol = np.round(sol.x, 4)

    is_new = True
    for u in unique_solutions:
        if np.allclose(u, rounded_sol, atol=1e-3):
            is_new = False
            break
    if is_new and sol.success:
        unique_solutions.append(rounded_sol)

    print(f"{str(guess):<20} | {str(sol.message):<15} | {rounded_sol.tolist()}")


for idx, state in enumerate(unique_solutions):
    print(f"State {idx+1}: {state}")

Initial Guess        | Solver Status   | Steady State [X1, X2, X3]
-----------------------------------------------------------------
[-1, -1, -1]         | The solution converged. | [-0.4583, -0.4583, -0.4583]
[-1, -1, 0]          | The solution converged. | [-0.4583, -0.4583, -0.4583]
[-1, -1, 1]          | The solution converged. | [-0.4583, -0.4583, -0.4583]
[-1, 0, -1]          | The solution converged. | [-0.4583, -0.4583, -0.4583]
[-1, 0, 0]           | The solution converged. | [-0.4583, -0.4583, -0.4583]
[-1, 0, 1]           | The solution converged. | [-0.0, -0.0, -0.0]
[-1, 1, -1]          | The solution converged. | [-0.4583, -0.4583, -0.4583]
[-1, 1, 0]           | The solution converged. | [0.0, 0.0, 0.0]
[-1, 1, 1]           | The solution converged. | [0.4583, 0.4583, 0.4583]
[0, -1, -1]          | The solution converged. | [-0.4583, -0.4583, -0.4583]
[0, -1, 0]           | The solution converged. | [-0.4583, -0.4583, -0.4583]
[0, -1, 1]           | The solution converge

In [39]:
print(root(lambda X: (1 - (1 + 0.1) / np.sqrt(1 + X**2)), 1))
print(root(lambda X: (1 - (1 + 0.1) / np.sqrt(1 + X**2)), -1))

 message: The solution converged.
 success: True
  status: 1
     fun: [ 0.000e+00]
       x: [ 4.583e-01]
  method: hybr
    nfev: 10
    fjac: [[-1.000e+00]]
       r: [-3.787e-01]
     qtf: [ 4.916e-13]
 message: The solution converged.
 success: True
  status: 1
     fun: [ 2.220e-16]
       x: [-4.583e-01]
  method: hybr
    nfev: 10
    fjac: [[-1.000e+00]]
       r: [ 3.787e-01]
     qtf: [ 4.916e-13]


In [44]:
# All 6
def system_dynamics(T, Y, epsilons, kappa, zeta):
  X1, X2, X3, V1, V2, V3 = Y
  eps1, eps2, eps3 = epsilons

  dX1 = V1
  dX2 = V2
  dX3 = V3

  dV1 = (
      -2 * zeta * V1
      - 2 * (1 - (1 + eps1) / np.sqrt(1 + X1**2)) * X1
      - kappa * (X1 - X2)
  )
  dV2 = (
      -2 * zeta * V2
      - 2 * (1 - (1 + eps2) / np.sqrt(1 + X2**2)) * X2
      - kappa * (X2 - X1)
      - kappa * (X2 - X3)
  )
  dV3 = (
      -2 * zeta * V3
      - 2 * (1 - (1 + eps3) / np.sqrt(1 + X3**2)) * X3
      - kappa * (X3 - X2)
  )

  return [dX1, dX2, dX3, dV1, dV2, dV3]


epsilons = [0.1, 0.1, 0.1]
kappa = 1.0
kappa_wall = 0.0
zeta = 0.5

initial_guesses = [
    [0.8, -0.8, 0.5],
    [-0.7, 0.6, -0.5],
    [0.2, 0.1, -0.3],
    [0.9, 0.9, 0.9],
    [-0.9, -0.9, -0.9],
]

fig = go.Figure()

T_span = (0, 15)
T_eval = np.linspace(0, 15, 300)

for idx, pos_guess in enumerate(initial_guesses):
  Y0 = pos_guess + [0.0, 0.0, 0.0]
  sol = solve_ivp(
      system_dynamics,
      T_span,
      Y0,
      args=(epsilons, kappa, zeta),
      t_eval=T_eval,
  )

  x1_path = sol.y[0]
  x2_path = sol.y[1]
  x3_path = sol.y[2]

  fig.add_trace(
      go.Scatter3d(
          x=x1_path,
          y=x2_path,
          z=x3_path,
          mode="lines+markers",
          marker=dict(size=3),
          line=dict(width=4),
          name=f"Trajectory {idx+1} (Start: {pos_guess})",
      )
  )

# Steady State point
# got to look x coord as one the y as one and z as one
steady_states_x = [-0.4583, 0.0, 0.4583]
steady_states_y = [-0.4583, 0.0, 0.4583]
steady_states_z = [-0.4583, 0.0, 0.4583]

fig.add_trace(
    go.Scatter3d(
        x=steady_states_x,
        y=steady_states_y,
        z=steady_states_z,
        mode="markers+text",
        marker=dict(size=10, color=["red", "black", "green"], symbol="diamond"),
        text=[
            "Stable Well (All Down)",
            "Unstable Peak (Center)",
            "Stable Well (All Up)",
        ],
        textposition="top center",
        name="Steady States",
    )
)

fig.update_layout(
    title=(
        "3D Phase-Space Trajectories: How All 3 Nodes Relax Into Steady States"
    ),
    scene=dict(
        xaxis_title="Node 1 Displacement ($X_1$)",
        yaxis_title="Node 2 Displacement ($X_2$)",
        zaxis_title="Node 3 Displacement ($X_3$)",
    ),
    width=950,
    height=750,
)

fig.show()

Interactive 3D phase-space plot saved as 'phase_space_3d.html'!


In [47]:
# All 6
def system_dynamics(T, Y, epsilons, kappa, zeta):
    X1, X2, X3, V1, V2, V3 = Y
    eps1, eps2, eps3 = epsilons

    dX1 = V1
    dX2 = V2
    dX3 = V3

    dV1 = (
        -2 * zeta * V1
        - 2 * (1 - (1 + eps1) / np.sqrt(1 + X1**2)) * X1
        - kappa * (X1 - X2)
    )
    dV2 = (
        -2 * zeta * V2
        - 2 * (1 - (1 + eps2) / np.sqrt(1 + X2**2)) * X2
        - kappa * (X2 - X1)
        - kappa * (X2 - X3)
    )
    dV3 = (
        -2 * zeta * V3
        - 2 * (1 - (1 + eps3) / np.sqrt(1 + X3**2)) * X3
        - kappa * (X3 - X2)
    )

    return [dX1, dX2, dX3, dV1, dV2, dV3]


epsilons = [0.1, 0.1, 0.1]
kappa = 1.0
kappa_wall = 0.0
zeta = 0.5

initial_guesses = []
for x in [-1, 0, 1]:
    for y in [-1, 0, 1]:
        for z in [-1, 0, 1]:
            initial_guesses.append([x, y, z])

fig = go.Figure()

T_span = (0, 15)
T_eval = np.linspace(0, 15, 300)

for idx, pos_guess in enumerate(initial_guesses):
    Y0 = pos_guess + [0.0, 0.0, 0.0]
    sol = solve_ivp(
        system_dynamics,
        T_span,
        Y0,
        args=(epsilons, kappa, zeta),
        t_eval=T_eval,
    )

    x1_path = sol.y[0]
    x2_path = sol.y[1]
    x3_path = sol.y[2]

    fig.add_trace(
        go.Scatter3d(
            x=x1_path,
            y=x2_path,
            z=x3_path,
            mode="lines+markers",
            marker=dict(size=3),
            line=dict(width=4),
            name=f"Trajectory {idx+1} (Start: {pos_guess})",
        )
    )

# Steady State point
# got to look x coord as one the y as one and z as one
steady_states_x = [-0.4583, 0.0, 0.4583]
steady_states_y = [-0.4583, 0.0, 0.4583]
steady_states_z = [-0.4583, 0.0, 0.4583]

fig.add_trace(
    go.Scatter3d(
        x=steady_states_x,
        y=steady_states_y,
        z=steady_states_z,
        mode="markers+text",
        marker=dict(size=10, color=["red", "black", "green"], symbol="diamond"),
        text=[
            "Stable Well (All Down)",
            "Unstable Peak (Center)",
            "Stable Well (All Up)",
        ],
        textposition="top center",
        name="Steady States",
    )
)

fig.update_layout(
    title=("3D Phase-Space Trajectories: How All 3 Nodes Relax Into Steady States"),
    scene=dict(
        xaxis_title="Node 1 Displacement ($X_1$)",
        yaxis_title="Node 2 Displacement ($X_2$)",
        zaxis_title="Node 3 Displacement ($X_3$)",
    ),
    width=950,
    height=750,
)

fig.show()

Interactive 3D phase-space plot saved as 'phase_space_3d.html'!


In [46]:
# Fade the wall so no kappa wall
def analyze_steady_state(X_steady, epsilons, kappa, zeta):
    X1, X2, X3 = X_steady
    eps1, eps2, eps3 = epsilons

    # Local stiffness derivative for each node
    k1 = -2 * (1 - (1 + eps1) / (1 + X1**2) ** 1.5)
    k2 = -2 * (1 - (1 + eps2) / (1 + X2**2) ** 1.5)
    k3 = -2 * (1 - (1 + eps3) / (1 + X3**2) ** 1.5)

    # This is stiffness matrix
    K_eff = np.array(
        [
            [k1 - kappa, kappa, 0],
            [kappa, k2 - 2 * kappa, kappa],
            [0, kappa, k3 - kappa],
        ]
    )

    # Build the full 6x6 Jacobian matrice since We have velcoty in this phase space too so there are 6 in this matrix
    # Top-left: 0, Top-right: Identity
    # Bottom-left: K_eff, Bottom-right: -2*zeta * Identity
    J = np.block(
        [
            [np.zeros((3, 3)), np.eye(3)],
            [K_eff, -2 * zeta * np.eye(3)],
        ]
    )

    eigenvalues = np.linalg.eigvals(J)

    is_stable = all(np.real(eigenvalues) < 0)

    return is_stable, eigenvalues


epsilons = [0.1, 0.1, 0.1]
kappa = 1.0
zeta = 0.5

states = {
    "Center Peak": [0.0, 0.0, 0.0],
    "All Down Well": [-0.4583, -0.4583, -0.4583],
    "All Up Well": [0.4583, 0.4583, 0.4583],
}

print(f"{'State Name':<15} | {'Is Stable?':<10} | {'Max Real Eigenvalue'}")
print("-" * 55)
for name, state in states.items():
    stable, evals = analyze_steady_state(state, epsilons, kappa, zeta)
    max_real = np.max(np.real(evals))
    print(f"{name:<15} | {str(stable):<10} | {max_real:.4f}")

State Name      | Is Stable? | Max Real Eigenvalue
-------------------------------------------------------
Center Peak     | False      | 0.1708
All Down Well   | True       | -0.5000
All Up Well     | True       | -0.5000


In [48]:
def check_controllability(X_steady, epsilons, kappa, zeta, B_matrix):
    X1, X2, X3 = X_steady
    eps1, eps2, eps3 = epsilons

    k1 = -2 * (1 - (1 + eps1) / (1 + X1**2) ** 1.5)
    k2 = -2 * (1 - (1 + eps2) / (1 + X2**2) ** 1.5)
    k3 = -2 * (1 - (1 + eps3) / (1 + X3**2) ** 1.5)

    K_eff = np.array(
        [
            [k1 - kappa, kappa, 0],
            [kappa, k2 - 2 * kappa, kappa],
            [0, kappa, k3 - kappa],
        ]
    )

    A = np.block([[np.zeros((3, 3)), np.eye(3)], [K_eff, -2 * zeta * np.eye(3)]])

    # C = [B, AB, A^2B, A^3B, A^4B, A^5B] n-1 power here
    n = A.shape[0]
    C = B_matrix
    current_term = B_matrix
    for i in range(1, n):
        current_term = A @ current_term
        C = np.hstack((C, current_term))

    rank = np.linalg.matrix_rank(C)
    is_controllable = rank == n
    return is_controllable, rank


epsilons = [0.1, 0.1, 0.1]
kappa = 1.0
zeta = 0.5

# Only push on node 1 or pushon V1 or accel on node 1
B_matrix = np.array([[0], [0], [0], [1], [0], [0]])

states = {
    "Center Peak": [0.0, 0.0, 0.0],
    "All Down Well": [-0.4583, -0.4583, -0.4583],
    "All Up Well": [0.4583, 0.4583, 0.4583],
}

# --- Run Controllability Analysis for Each State ---
print(f"{'Steady State':<15} | {'Matrix Rank':<12} | {'Fully Controllable?'}")
print("-" * 45)

for name, state in states.items():
    controllable, rank = check_controllability(state, epsilons, kappa, zeta, B_matrix)
    print(f"{name:<15} | {rank:<12} | {str(controllable)}")

Steady State    | Matrix Rank  | Fully Controllable?
---------------------------------------------
Center Peak     | 6            | True
All Down Well   | 6            | True
All Up Well     | 6            | True
